In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = "data/messy_mashup/genres_stems" # Enter dataset path
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"] # Make the list of all genres available
STEMS = {"drums.wav", "vocals.wav", "bass.wav", "other.wav"} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [4]:
import math

def build_dataset_unused(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)
    def add_songs_to_dict(target_dict, genre, song_dirs):
        for song_dir in song_dirs:
            for stem_filename in STEMS:
                stem_key = stem_filename.replace('.wav','')
                path = os.path.join(song_dir, stem_filename)
                target_dict[genre][stem_key].append(path)
    for g in GENRES:
        gdir = os.path.join(root_dir, g)
        songs = []
        for entry in sorted(os.listdir(gdir)):
            songs.append(os.path.join(gdir, entry))
        rng.shuffle(songs)
        n_val = int(math.floor(len(songs) * val_split)) # 17
        val_songs = songs[:n_val]
        train_songs = songs[n_val:]
        add_songs_to_dict(train_dataset, g, train_songs)
        add_songs_to_dict(val_dataset, g, val_songs)

    return train_dataset, val_dataset

In [5]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)

    def add_songs_to_dict(target, genre, song_dirs):
        for song_dir in song_dirs:
            for stem in STEMS:
                key = stem.replace('.wav','')
                target[genre][key].append(os.path.join(song_dir, stem))

    for genre in GENRES:
        gdir = os.path.join(root_dir, genre)
        if not os.path.isdir(gdir):
            continue
        song_dirs = [os.path.join(gdir, e) for e in sorted(os.listdir(gdir))]

        songs = []
        for sd in song_dirs:
            ok = True
            for stem in STEMS:
                p = os.path.join(sd, stem)
                if not os.path.isfile(p):
                    ok = False
                    break
                if os.path.getsize(p) < (4 * 1024):
                    ok = False
                    break
            if ok:
                songs.append(sd)

        rng.shuffle(songs)

        n_val = int(math.floor(len(songs) * val_split))
        val_songs = songs[:n_val]
        train_songs = songs[n_val:]

        add_songs_to_dict(train_dataset, genre, train_songs)
        add_songs_to_dict(val_dataset, genre, val_songs)

    return train_dataset, val_dataset

In [ ]:
tr, val = build_dataset(DATA_ROOT)
tr["rock"]["drums"][:5]

['data/messy_mashup/genres_stems/rock/rock.00082/drums.wav',
 'data/messy_mashup/genres_stems/rock/rock.00013/drums.wav',
 'data/messy_mashup/genres_stems/rock/rock.00089/drums.wav',
 'data/messy_mashup/genres_stems/rock/rock.00096/drums.wav',
 'data/messy_mashup/genres_stems/rock/rock.00008/drums.wav']

In [ ]:
# Q2: What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB?
!find data/messy_mashup/genres_stems -type f -size +5294365c | wc -l; find data/messy_mashup/genres_stems -type f -size -5294575c | wc -l;
print(3816 - 2744)

2744
3816
1072


In [ ]:
# Q3: What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?
len(tr["reggae"]["drums"]) - len(val["country"]["vocals"])

66

In [10]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []

    for genre, stems_dict in dataset_dict.items():
        print(genre)
        for stem_name, paths in stems_dict.items():
            for file_path in paths:
                y, _ = librosa.load(file_path, sr=sr)
                total_len = len(y)
                total_duration = total_len / float(sr)

                intervals = librosa.effects.split(y, top_db=top_db, hop_length=HOP_LENGTH) # (N, 2) or empty
                if intervals.size == 0:
                    max_silence = total_duration
                    silence_types = ["full"] if max_silence >= threshold_sec else []
                else:
                    starts = intervals[:, 0]
                    ends = intervals[:, 1]
                    silence_durations = []
                    silence_types_full = []  # collect all silence durations with labels

                    # start
                    if starts[0] > 0:
                        dur = starts[0] / float(sr)
                        silence_durations.append(dur)
                        silence_types_full.append(("start", dur))

                    # middle
                    for i in range(len(intervals) - 1):
                        gap = (starts[i+1] - ends[i]) / float(sr)
                        if gap > 0:
                            silence_durations.append(gap)
                            silence_types_full.append(("middle", gap))

                    # end
                    if ends[-1] < total_len:
                        dur = (total_len - ends[-1]) / float(sr)
                        silence_durations.append(dur)
                        silence_types_full.append(("end", dur))

                    max_silence = max(silence_durations) if silence_durations else 0.0

                    # collect only locations whose silence >= threshold_sec
                    silence_types = [label for (label, dur) in silence_types_full if dur >= threshold_sec]

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_types) if len(silence_types) > 0 else "",
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records, columns=["Genre", "Stem", "Duration", "Max_Silence_Sec", "Silence_Location", "File_Path"])
    return df

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

blues
classical
country
disco
hiphop
jazz
metal
pop
reggae
rock


In [11]:
df_silence

,Genre,Stem,Duration,Max_Silence_Sec,Silence_Location,File_Path
0,blues,other,30.01,6.69,middle,data/messy_mashup/genres_stems/blues/blues.000...
1,blues,other,30.01,5.12,end,data/messy_mashup/genres_stems/blues/blues.000...
2,blues,other,30.01,5.60,"middle, middle",data/messy_mashup/genres_stems/blues/blues.000...
3,blues,other,30.01,6.49,end,data/messy_mashup/genres_stems/blues/blues.000...
4,blues,drums,30.01,16.28,middle,data/messy_mashup/genres_stems/blues/blues.000...
...,...,...,...,...,...,...
683,rock,vocals,30.01,8.22,middle,data/messy_mashup/genres_stems/rock/rock.00000...
684,rock,vocals,30.01,5.55,"middle, middle",data/messy_mashup/genres_stems/rock/rock.00009...
685,rock,vocals,30.01,18.65,middle,data/messy_mashup/genres_stems/rock/rock.00033...
686,rock,vocals,30.01,7.57,middle,data/messy_mashup/genres_stems/rock/rock.00012...


In [26]:
# Q4: Total number of files having silence >= 5 secs
print(f"Q4: {len(df_silence)}")

# Q5: Total number of tracks in vocals where silence >= 5 secs
print(f"Q5: {df_silence.loc[df_silence["Stem"] == "vocals"].shape[0]}")

# Q6: Average silence length in vocals (in secs) -- use only vocals rows
print(f"Q6: {df_silence.loc[df_silence["Stem"] == "vocals", "Max_Silence_Sec"].mean()}")

# Q7: Total number of drums sound tracks in jazz where silence >= 5 secs
print(f"Q7: {df_silence.loc[(df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums")].shape[0]}")

# Q8: Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.
print(f"Q8: {df_silence.loc[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
].shape[0]}")

# Q9: Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10.
print(f"Q9: {df_silence.loc[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]}")

Q4: 688
Q5: 312
Q6: 12.544326923076923
Q7: 22
Q8: 7
Q9: 8


In [35]:
stems_audio = []
try:
    for key in STEM_KEYS:
      y, _ = librosa.load(tr[GENRE_TO_TEST][key][SONG_INDEX], sr=SR, duration=5.0)
      stems_audio.append(y)
    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [36]:
# Q10:
y.shape

(110250,)

In [37]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.stack(stems_audio, axis=0)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2, axis=0))
# Q11:
print(f"rms_val: {rms_val}")

#Peak Normalization
max_val = np.max(np.abs(mix_raw))
# Q12:
print(f"max_val: {max_val}")

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------


rms_val: 0.09343763440847397
max_val: 0.5610571503639221
